# Prometheus - Medical Image Segmentation Training on PUMA

**Dataset:** [PUMA](https://puma.grand-challenge.org/dataset/) — Panoptic segmentation of nUclei and tissue in MelanomA

**Models:** UNetTissue (6 tissue classes) | DualUNet (6 tissue + 10 nuclei classes)

**GPU:** Google Colab G4 / A100 100GB VRAM

---
## 1. Setup

In [ ]:
import os, sys, math, random, json
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

import numpy as np
import matplotlib.pyplot as plt
import cv2
import tifffile

print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
REPO_DIR = "/content/prometheus"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoangtung386/Prometheus.git {REPO_DIR}

%cd {REPO_DIR}
!pip install -e .
!pip install tensorboard matplotlib opencv-python-headless tifffile
sys.path.insert(0, REPO_DIR)

In [ ]:
from prometheus import UNetTissue, DualUNet, CombinedLoss
from prometheus.config import ModelConfig
from prometheus.data import PUMADataset, create_puma_dataloaders, train_transform, val_transform

---
## 2. Configuration

In [ ]:
class Config:
    # ── Model ──
    model_type: str = "DualUNet"           # "UNetTissue" | "DualUNet"
    in_chans: int = 3
    image_size: int = 1024                 # PUMA ROI resolution

    # tissue=6 classes, nuclei=10 classes (both exclude background=0)
    # UNetTissue → num_classes=6, DualUNet → needs 6 & 10 separately
    num_tissue_classes: int = 6
    num_nuclei_classes: int = 10

    # ── Training ──
    batch_size: int = 16                   # 100GB VRAM
    epochs: int = 200
    lr: float = 2e-4
    weight_decay: float = 1e-2
    warmup_epochs: int = 10
    grad_clip_norm: float = 1.0
    amp: bool = True

    # ── Data ──
    # PUMA download: https://zenodo.org/records/14869398
    # Unzip 01_training_dataset_tif_ROIs.zip → images/
    # Unzip 01_training_dataset_geojson_tissue.zip → geojson_tissue/
    # Unzip 01_training_dataset_geojson_nuclei.zip → geojson_nuclei/
    data_root: str = "/content/drive/MyDrive/puma_data"
    num_workers: int = 8
    val_split: float = 0.1

    # ── Logging ──
    log_dir: str = "/content/drive/MyDrive/prometheus_logs"
    ckpt_dir: str = "/content/drive/MyDrive/prometheus_checkpoints"
    log_interval: int = 10
    save_interval: int = 5

cfg = Config()
os.makedirs(cfg.log_dir, exist_ok=True)
os.makedirs(cfg.ckpt_dir, exist_ok=True)

print(f"Model: {cfg.model_type}")
print(f"Tissue classes: {cfg.num_tissue_classes}, Nuclei classes: {cfg.num_nuclei_classes}")
print(f"Image size: {cfg.image_size}x{cfg.image_size}, Batch: {cfg.batch_size}")

---
## 3. Dataset — PUMA (tissue 6 classes + nuclei 10 classes)

Dataset yêu cầu:
```
data_root/
├── images/              # 206 *.tif (1024x1024, 40x H&E)
├── geojson_tissue/      # 206 *_tissue.geojson (6 classes)
└── geojson_nuclei/      # 206 *_nuclei.geojson (10 classes)
```

**Tissue classes:** background(0), tumor(1), stroma(2), epidermis(3), necrosis(4), blood_vessel(5)

**Nuclei classes:** background(0), tumor(1), stroma(2), vascular_endothelium(3), histiocyte(4), melanophage(5), lymphocyte(6), plasma_cell(7), neutrophil(8), apoptotic_cell(9), epithelium(10)

In [ ]:
# ── Load PUMA dataset ──
if os.path.exists(cfg.data_root):
    train_loader, val_loader = create_puma_dataloaders(
        root=cfg.data_root,
        image_size=cfg.image_size,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        val_split=cfg.val_split,
    )
else:
    print(f"Data not found at {cfg.data_root}")
    print("Falling back to dummy data for testing...")
    train_loader = val_loader = None

### 3a. Quick visualization

In [ ]:
def visualize_sample(dataset, idx=0):
    img, targets = dataset[idx]
    tissue_mask = targets["tissue"]   # (6, H, W)
    nuclei_mask = targets["nuclei"]   # (11, H, W)

    tissue_overlay = tissue_mask.argmax(dim=0).numpy()
    nuclei_overlay = nuclei_mask.argmax(dim=0).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0))
    axes[0].set_title("Input (normalized)")
    axes[1].imshow(tissue_overlay, cmap="tab10", vmin=0, vmax=5)
    axes[1].set_title("Tissue (6 classes)")
    axes[2].imshow(nuclei_overlay, cmap="tab10", vmin=0, vmax=10)
    axes[2].set_title("Nuclei (10 classes)")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

if os.path.exists(cfg.data_root):
    ds = PUMADataset(root=cfg.data_root, image_size=cfg.image_size)
    visualize_sample(ds, 0)

### 3b. In-memory mask caching
Để tránh render GeoJSON polygon → mask mỗi epoch, dataset đã cache mask trong RAM lần đầu.
Với 206 ROI × (6 + 11) channels × 1024² ≈ 3.6 GB — cần heap RAM > 12 GB (Colab G4 OK).
Nếu OOM → tắt cache: `PUMADataset(..., cache_masks=False)`

---
## 4. Model

### Lưu ý về số classes:
- **UNetTissue** nhận `num_classes` duy nhất → chỉ train tissue với 6 classes
- **DualUNet** hiện tại dùng chung `num_classes` cho cả 2 head → cần custom model cho 6 & 10
  → Tạm thời train từng head riêng, hoặc modify model sau.

In [ ]:
if cfg.model_type == "UNetTissue":
    model_cfg = ModelConfig(
        in_chans=cfg.in_chans,
        num_classes=cfg.num_tissue_classes,  # 6 tissue classes
        drop_path_rate=0.1,
    )
    model = UNetTissue(config=model_cfg).to(device)
else:
    # DualUNet: tạm dùng nuclei classes cho cả 2 head
    # TODO: tách riêng tissue_head.num_classes và nuclei_head.num_classes
    model_cfg = ModelConfig(
        in_chans=cfg.in_chans,
        num_classes=cfg.num_nuclei_classes + 1,  # 11 channels (bg + 10 nuclei)
        drop_path_rate=0.1,
    )
    model = DualUNet(config=model_cfg).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{cfg.model_type}: {total:,} params ({trainable:,} trainable)")

---
## 5. Optimizer & Scheduler

In [ ]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    betas=(0.9, 0.999), eps=1e-8,
)

def warmup_cosine_lr(epoch: int) -> float:
    if epoch < cfg.warmup_epochs:
        return epoch / max(1, cfg.warmup_epochs)
    progress = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine_lr)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
writer = SummaryWriter(log_dir=cfg.log_dir)
criterion = CombinedLoss(bce_weight=1.0, dice_weight=1.0)
print("Ready: optimizer, scheduler, AMP scaler, TensorBoard")

---
## 6. Training Loop

In [ ]:
def dice_score(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    pred = pred.argmax(dim=1)                          # (B, H, W) class indices
    pred = nn.functional.one_hot(pred, num_classes=target.shape[1]).permute(0, 3, 1, 2)
    intersection = (pred * target).sum(dim=(2, 3))
    cardinality = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return ((2 * intersection + eps) / (cardinality + eps)).mean(dim=1)


def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch, cfg):
    model.train()
    total_loss = 0.0
    n = len(loader)

    for batch_idx, (images, targets) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        t_mask = targets["tissue"].to(device, non_blocking=True)
        n_mask = targets["nuclei"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.amp):
            if cfg.model_type == "UNetTissue":
                logits = model(images)
                loss = criterion(logits, t_mask)
            else:
                pred_t, pred_n, moe_loss = model(images)
                loss = criterion(pred_t, t_mask) + criterion(pred_n, n_mask) + 0.01 * moe_loss

        scaler.scale(loss).backward()
        if cfg.grad_clip_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if batch_idx % cfg.log_interval == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"E{epoch:03d} B{batch_idx:04d}/{n}  loss={loss.item():.4f}  lr={lr_now:.2e}")

    return total_loss / n


@torch.no_grad()
def validate(model, loader, criterion, cfg):
    model.eval()
    total_loss = 0.0
    dice_tissue, dice_nuclei = [], []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        t_mask = targets["tissue"].to(device, non_blocking=True)
        n_mask = targets["nuclei"].to(device, non_blocking=True)

        if cfg.model_type == "UNetTissue":
            logits = model(images)
            loss = criterion(logits, t_mask)
            dice_tissue.append(dice_score(logits, t_mask))
        else:
            pred_t, pred_n, _ = model(images)
            loss = criterion(pred_t, t_mask) + criterion(pred_n, n_mask)
            dice_tissue.append(dice_score(pred_t, t_mask))
            dice_nuclei.append(dice_score(pred_n, n_mask))

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    avg_dice_t = torch.cat(dice_tissue).mean().item() if dice_tissue else 0.0
    avg_dice_n = torch.cat(dice_nuclei).mean().item() if dice_nuclei else 0.0
    return avg_loss, avg_dice_t, avg_dice_n

In [ ]:
if train_loader is None:
    print("=== DUMMY DATA MODE ===")
    dummy = [(torch.randn(3, 256, 256),
              {"tissue": torch.randint(0, 2, (6, 256, 256)).float(),
               "nuclei": torch.randint(0, 2, (11, 256, 256)).float()})
             for _ in range(64)]
    train_loader = val_loader = DataLoader(dummy, batch_size=4, shuffle=True)
    cfg.epochs = 5

best_dice = 0.0
start_epoch = 0

for epoch in range(start_epoch, cfg.epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch, cfg)
    scheduler.step()

    val_loss, val_dice_t, val_dice_n = validate(model, val_loader, criterion, cfg)

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("Dice/tissue", val_dice_t, epoch)
    writer.add_scalar("Dice/nuclei", val_dice_n, epoch)
    writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

    print(f"─── E{epoch:03d}  train={train_loss:.4f}  val={val_loss:.4f}  "
          f"dice_t={val_dice_t:.4f}  dice_n={val_dice_n:.4f}  "
          f"lr={optimizer.param_groups[0]['lr']:.2e}")

    monitor = val_dice_t + val_dice_n if val_dice_n > 0 else val_dice_t
    if monitor > best_dice:
        best_dice = monitor
        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_dice": best_dice,
            "config": cfg,
        }
        torch.save(ckpt, os.path.join(cfg.ckpt_dir, f"{cfg.model_type}_best.pth"))
        print(f"  → Saved best (dice_t={val_dice_t:.4f}, dice_n={val_dice_n:.4f})")

    if (epoch + 1) % cfg.save_interval == 0:
        torch.save(ckpt, os.path.join(cfg.ckpt_dir, f"{cfg.model_type}_epoch{epoch:03d}.pth"))

writer.close()
print(f"\nDone! Best monitor dice: {best_dice:.4f}")

---
## 7. Inference & Visualization

In [ ]:
@torch.no_grad()
def predict_sample(model, image_tensor, model_type):
    model.eval()
    inp = image_tensor.unsqueeze(0).to(device)
    if model_type == "UNetTissue":
        logits = model(inp)
        return logits.argmax(dim=1).squeeze(0).cpu().numpy()
    else:
        t_logits, n_logits, _ = model(inp)
        return {
            "tissue": t_logits.argmax(dim=1).squeeze(0).cpu().numpy(),
            "nuclei": n_logits.argmax(dim=1).squeeze(0).cpu().numpy(),
        }


def show_prediction(model, dataset, idx=0, model_type="DualUNet"):
    img, targets = dataset[idx]
    pred = predict_sample(model, img, model_type)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes[0, 0].imshow(img.permute(1, 2, 0))
    axes[0, 0].set_title("Input")

    if model_type == "UNetTissue":
        axes[0, 1].imshow(targets["tissue"].argmax(dim=0), cmap="tab10", vmin=0, vmax=5)
        axes[0, 1].set_title("Tissue GT")
        axes[0, 2].imshow(pred, cmap="tab10", vmin=0, vmax=5)
        axes[0, 2].set_title("Tissue Pred")
    else:
        axes[0, 1].imshow(targets["tissue"].argmax(dim=0), cmap="tab10", vmin=0, vmax=5)
        axes[0, 1].set_title("Tissue GT")
        axes[0, 2].imshow(pred["tissue"], cmap="tab10", vmin=0, vmax=5)
        axes[0, 2].set_title("Tissue Pred")
        axes[1, 1].imshow(targets["nuclei"].argmax(dim=0), cmap="tab10", vmin=0, vmax=10)
        axes[1, 1].set_title("Nuclei GT")
        axes[1, 2].imshow(pred["nuclei"], cmap="tab10", vmin=0, vmax=10)
        axes[1, 2].set_title("Nuclei Pred")
        axes[1, 0].axis("off")

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


if os.path.exists(cfg.data_root):
    ds = PUMADataset(root=cfg.data_root, image_size=cfg.image_size, augment=False)
    show_prediction(model, ds, 0, cfg.model_type)

---
## 8. Data Preprocessing Utilities (dùng riêng)

In [ ]:
# GeoJSON → mask rendering
from prometheus.data.puma_dataset import (
    geojson_to_mask,
    TISSUE_CLASSES, NUCLEI_CLASSES,
    TISSUE_CLASS_TO_IDX, NUCLEI_CLASS_TO_IDX,
)

# Augmentations
from prometheus.data.transforms import (
    train_transform, val_transform,
    RandomHorizontalFlip, RandomRotate90,
    ElasticDeformation, NormalizeTile,
)

print("Available utilities:")
print(f"  Tissue classes ({len(TISSUE_CLASSES)}): {TISSUE_CLASSES}")
print(f"  Nuclei classes ({len(NUCLEI_CLASSES)}): {NUCLEI_CLASSES}")
print(f"  train_transform: flip + rotate + elastic + brightness + normalize")

---
## 9. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {cfg.log_dir}